# HTRU2 Pulsar — KAN + PDE约束 + CIM 实时量子刷新
## Parkes 64m 射电望远镜 → PCA 2D → Poisson PDE → 能量泛函 QUBO
### 每 50 epoch 调一次 CIM 真机刷新量子约束 (500轮共10次)
量子从离线预设变为在线协处理器

In [1]:
import io, zipfile, requests
import numpy as np
import torch, torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd, warnings, json, os, time

import kaiwu as kw
kw.common.CheckpointManager.save_dir = '/tmp'

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = 'D:/QPDE/photo+kan/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

BIT_WIDTH = 8
CIM_TIMEOUT = 600
CIM_POLL = 5
print('模块加载完成')

模块加载完成


In [2]:
# ========== RBF-KAN ==========
class KANLayer(nn.Module):
    def __init__(self, in_dim, out_dim, grid_size=8):
        super().__init__()
        self.register_buffer('centers', torch.linspace(-1, 1, grid_size))
        self.width = nn.Parameter(torch.ones(1) * 2.0 / grid_size)
        self.coef = nn.Parameter(torch.randn(out_dim, in_dim, grid_size) * 0.1)
        self.base = nn.Linear(in_dim, out_dim)
    def forward(self, x):
        x_exp = x.unsqueeze(-1); c = self.centers.view(1, 1, -1)
        rbf = torch.exp(-((x_exp - c) / self.width)**2)
        return torch.einsum('big,oig->bo', rbf, self.coef) + self.base(x)

class KAN(nn.Module):
    def __init__(self, layers, grid_size=8):
        super().__init__()
        self.layers = nn.ModuleList([KANLayer(layers[i], layers[i+1], grid_size) for i in range(len(layers)-1)])
    def forward(self, x):
        for l in self.layers[:-1]: x = torch.tanh(l(x))
        return self.layers[-1](x)

print('KAN 模型已定义')

KAN 模型已定义


In [3]:
# ========== 加载 HTRU2 脉冲星数据 ==========
print('=== 加载 HTRU2 脉冲星数据 ===')
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00372/HTRU2.zip'
zf = zipfile.ZipFile(io.BytesIO(requests.get(url).content))
df = pd.read_csv(zf.open('HTRU_2.csv'), header=None,
    names=['mean_ip','std_ip','excess_kurtosis_ip','skewness_ip',
           'mean_dm','std_dm','excess_kurtosis_dm','skewness_dm','class'])
X_all = df.drop(columns=['class']).values.astype(float)
y_all = np.where(df['class'].values == 1, 1, -1)  # pulsar=+1, RFI=-1
print(f'全部: {len(y_all)} 条, 脉冲星={(y_all==1).sum()}, RFI={(y_all==-1).sum()}')

rng = np.random.default_rng(42)
idx = rng.choice(len(y_all), 2000, replace=False)
X_s, y_s = X_all[idx], y_all[idx]
print(f'采样: {len(y_s)} 条')

=== 加载 HTRU2 脉冲星数据 ===
全部: 17898 条, 脉冲星=1639, RFI=16259
采样: 2000 条


In [4]:
# ========== PCA 降维 2D + 缩放到 [0.15, 0.85] ==========
X_2d = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X_s))
for j in range(2):
    lo, hi = X_2d[:,j].min(), X_2d[:,j].max()
    X_2d[:,j] = (X_2d[:,j] - lo) / (hi - lo) * 0.7 + 0.15

In [5]:
# ========== 9×9 网格 FEM ==========
N = 9; h = 1.0/(N-1)
GX, GY = np.meshgrid(np.linspace(0,1,N), np.linspace(0,1,N), indexing='ij')
nodes = np.column_stack([GX.ravel(), GY.ravel()])
boundary = (GX.ravel()==0)|(GX.ravel()==1)|(GY.ravel()==0)|(GY.ravel()==1)
internal = ~boundary; n_i = internal.sum()
print(f'网格: {N}×{N}={N*N} 节点, 内部={n_i}, QUBO={n_i*BIT_WIDTH} bit')

# 高斯核源项: f(node) = Σ y_k * exp(-||node - p_k||²/(2σ²))
sigma = 0.12
f = np.zeros(N*N)
for k in range(len(y_s)):
    f += y_s[k] * np.exp(-np.sum((nodes - X_2d[k])**2, axis=1) / (2*sigma**2))

# 5点差分刚度矩阵
idx_map = -np.ones(N*N, dtype=int); idx_map[internal] = np.arange(n_i)
K_ii = np.zeros((n_i, n_i))
for i in range(1, N-1):
    for j in range(1, N-1):
        ki = idx_map[i*N+j]; K_ii[ki, ki] = 4.0
        for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
            nk = (i+di)*N + (j+dj)
            if internal[nk]: K_ii[ki, idx_map[nk]] = -1.0
rhs = h**2 * f[internal]

cond_K = np.linalg.cond(K_ii)
print(f'K_ii: {K_ii.shape}, κ(K)={cond_K:.2e}, SPD={np.allclose(K_ii, K_ii.T)}')

网格: 9×9=81 节点, 内部=49, QUBO=392 bit
K_ii: (49, 49), κ(K)=2.53e+01, SPD=True


In [6]:
# ========== 经典参考解 & 变量上下界 ==========
u_ref_i = np.linalg.solve(K_ii, rhs)
u_ref = np.zeros(N*N); u_ref[internal] = u_ref_i
margin = 0.5; rng_val = max(u_ref_i.max()-u_ref_i.min(), 0.1)
lb = u_ref_i.min() - margin * rng_val
ub = u_ref_i.max() + margin * rng_val
print(f'参考解: [{u_ref_i.min():.4f}, {u_ref_i.max():.4f}], 编码界: [{lb:.4f}, {ub:.4f}]')

参考解: [-30.3658, -1.0726], 编码界: [-45.0124, 13.5740]


In [7]:
# ========== 能量泛函 QUBO 构建 (预计算, 训练期间复用) ==========
def build_energy_qubo(K, r, bw, lb, ub):
    """能量泛函: E(u) = 1/2 u^T K u - u^T r"""
    n = K.shape[0]; nvar = n * bw
    scale = (ub - lb) / (2**bw - 1)
    s = scale * np.array([2**k for k in range(bw)])
    ssT = np.outer(s, s); c = lb * np.ones(n); w = K @ c - r
    Q = np.zeros((nvar, nvar))
    for i in range(n):
        for j in range(i, n):
            aij = K[i,j]
            if abs(aij) < 1e-15: continue
            ri, rj = i*bw, j*bw
            blk = 0.5 * aij * ssT
            Q[ri:ri+bw, rj:rj+bw] += blk
            if i != j: Q[rj:rj+bw, ri:ri+bw] += blk.T
    for i in range(n):
        Q[i*bw:(i+1)*bw, i*bw:(i+1)*bw] += np.diag(w[i] * s)
    return Q.astype(np.float32), nvar, scale

Q_float, nvar, scale = build_energy_qubo(K_ii, rhs, BIT_WIDTH, lb, ub)
Q_qubo = kw.qubo.adjust_qubo_matrix_precision(Q_float)
ising_mat, ising_bias = kw.conversion.qubo_matrix_to_ising_matrix(Q_qubo)
ising_model = kw.ising.IsingModel(
    variables=[f'x[{i}]' for i in range(ising_mat.shape[0])],
    ising_matrix=ising_mat, bias=ising_bias)
print(f'能量QUBO: {nvar}×{nvar}, K_ii条件数={cond_K:.1f}, Ising: {ising_mat.shape}')

能量QUBO: 392×392, K_ii条件数=25.3, Ising: (393, 393)


In [8]:
# ========== CIM 求解函数 ==========
def cim_solve_once(task_name):
    """提交 + 阻塞等待 → 返回解码后的内部解"""
    opt = kw.cim.CIMOptimizer(task_name=task_name)
    opt.solve(ising_model.get_matrix())  # solve#1: 提交
    t0 = time.time()
    sol = None
    while time.time() - t0 < CIM_TIMEOUT:
        try:
            sol = opt.solve(ising_model.get_matrix())  # solve#2: 取回
            if sol is not None: break
        except Exception:
            pass
        time.sleep(CIM_POLL)
    if sol is None:
        raise RuntimeError(f'CIM {task_name} 超时')

    if sol.ndim == 2:
        sols_bin = (sol[:,:-1] * sol[:,-1:]+1)/2
        energies = np.array([z @ Q_qubo @ z for z in sols_bin])
        z_best = sols_bin[np.argmin(energies)]
    else:
        z_best = sol

    s = scale * np.array([2**k for k in range(BIT_WIDTH)])
    u_i = np.array([np.dot(s, z_best[i*BIT_WIDTH:(i+1)*BIT_WIDTH]) + lb for i in range(n_i)])
    return u_i

print('CIM 求解函数已定义')

CIM 求解函数已定义


In [9]:
# ========== 首次 CIM 量子求解 (初始预设) ==========
print('=== 首次 CIM 求解 (初始预设) ===')
u_quantum_i_init = cim_solve_once('htru2_q0')
u_quantum_init = np.zeros(N*N); u_quantum_init[internal] = u_quantum_i_init
rmse_init = np.sqrt(np.mean((u_quantum_i_init - u_ref_i)**2))
print(f'首次量子解 RMSE vs 经典: {rmse_init:.6e}')

=== 首次 CIM 求解 (初始预设) ===
[2026-05-21 20:44:28] [INFO    ] [kaiwu.cim._optimizer_adapter:5] - Task submit successfully, waiting for data validation. Task name: htru2_q0
[2026-05-21 20:44:28] [INFO    ] [kaiwu.cim._optimizer_adapter:10] - Task is still processing: htru2_q0
[2026-05-21 20:44:34] [INFO    ] [kaiwu.cim._optimizer_adapter:10] - Task is still processing: htru2_q0
[2026-05-21 20:44:39] [INFO    ] [kaiwu.cim._optimizer_adapter:10] - Task is still processing: htru2_q0
[2026-05-21 20:44:44] [INFO    ] [kaiwu.cim._optimizer_adapter:10] - Task is still processing: htru2_q0
[2026-05-21 20:44:49] [INFO    ] [kaiwu.cim._optimizer_adapter:10] - Task is still processing: htru2_q0
[2026-05-21 20:44:55] [INFO    ] [kaiwu.cim._optimizer_adapter:10] - Task is still processing: htru2_q0
[2026-05-21 20:45:00] [INFO    ] [kaiwu.cim._optimizer_adapter:10] - Task is still processing: htru2_q0
[2026-05-21 20:45:05] [INFO    ] [kaiwu.cim._optimizer_adapter:10] - Task is still processing: htru2_q0


In [10]:
# ========== 转为 tensor ==========
X_2d_t = torch.tensor(X_2d, dtype=torch.float32, device=DEVICE)
y_t = torch.tensor(y_s, dtype=torch.float32, device=DEVICE)
sigma_t = torch.tensor(sigma, dtype=torch.float32, device=DEVICE)
nodes_t = torch.tensor(nodes, dtype=torch.float32, device=DEVICE)
preset_xy = torch.tensor(nodes[internal], dtype=torch.float32, device=DEVICE)
u_ref_t = torch.tensor(u_ref, dtype=torch.float32, device=DEVICE).view(-1,1)

# 样本点 tensor (用于分类准确率评估)
sample_xy_t = torch.tensor(X_2d, dtype=torch.float32, device=DEVICE)

# 首次量子解作为初始预设
preset_u = torch.tensor(u_quantum_init[internal], dtype=torch.float32, device=DEVICE).view(-1,1)

def source_fn(xy):
    diff = xy.unsqueeze(1) - X_2d_t.unsqueeze(0)
    dist2 = (diff**2).sum(dim=-1)
    return (y_t.unsqueeze(0) * torch.exp(-dist2 / (2*sigma_t**2))).sum(dim=-1)

print('tensor 转换完成')

tensor 转换完成


In [14]:
import logging
logging.getLogger('kaiwu').setLevel(logging.WARNING)
# ========== KAN + PDE + CIM 实时训练 ==========
print('\n=== KAN + PDE + CIM 实时训练 (每50 epoch 刷新量子约束) ===')
model = KAN([2, 64, 64, 1], grid_size=8).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
N_col, N_bc, N_epoch = 1024, 256, 500
CIM_INTERVAL = 50
w_pde, w_bc, w_preset = 1.0, 10.0, 8.0

cim_count = 1
cim_rmse_history = [float(rmse_init)]  # 每次 CIM 返回的 RMSE
hist = {'loss':[], 'l2':[], 'acc':[], 'cim_epochs':[0]}
t0 = time.time()

for epoch in range(N_epoch):
    # --- 每 50 epoch 调 CIM 刷新量子约束 ---
    if epoch > 0 and epoch % CIM_INTERVAL == 0:
        print(f'  epoch {epoch}: 调用 CIM 刷新量子约束 (第 {cim_count+1}/10 次)...')
        t_cim = time.time()
        u_qi = cim_solve_once(f'htru2_q{epoch}')
        u_q = np.zeros(N*N); u_q[internal] = u_qi
        rmse_now = np.sqrt(np.mean((u_qi - u_ref_i)**2))
        preset_u = torch.tensor(u_q[internal], dtype=torch.float32, device=DEVICE).view(-1,1)
        cim_count += 1
        cim_rmse_history.append(float(rmse_now))
        hist['cim_epochs'].append(epoch)
        print(f'    完成, RMSE vs 经典={rmse_now:.4e},')

    # --- PDE + BC ---
    col = torch.rand(N_col, 2, device=DEVICE, requires_grad=True)
    bp = torch.rand(N_bc, 1, device=DEVICE); bs = torch.randint(0, 4, (N_bc,), device=DEVICE)
    bc_xy = torch.zeros(N_bc, 2, device=DEVICE)
    bc_xy[bs==0,0]=0; bc_xy[bs==0,1]=bp[bs==0,0]
    bc_xy[bs==1,0]=1; bc_xy[bs==1,1]=bp[bs==1,0]
    bc_xy[bs==2,0]=bp[bs==2,0]; bc_xy[bs==2,1]=0
    bc_xy[bs==3,0]=bp[bs==3,0]; bc_xy[bs==3,1]=1

    u_col = model(col)
    grad = torch.autograd.grad(u_col, col, torch.ones_like(u_col), create_graph=True)[0]
    u_x, u_y = grad[:,0:1], grad[:,1:2]
    u_xx = torch.autograd.grad(u_x, col, torch.ones_like(u_x), create_graph=True)[0][:,0:1]
    u_yy = torch.autograd.grad(u_y, col, torch.ones_like(u_y), create_graph=True)[0][:,1:2]
    f_val = source_fn(col)

    loss_pde = (-u_xx - u_yy - f_val.view(-1,1)).pow(2).mean()
    loss_bc = model(bc_xy).pow(2).mean()
    loss_preset = (model(preset_xy) - preset_u).pow(2).mean()
    loss = w_pde*loss_pde + w_bc*loss_bc + w_preset*loss_preset

    opt.zero_grad(); loss.backward(); opt.step()
    hist['loss'].append(loss.item())

    if epoch % 100 == 0:
        model.eval()
        with torch.no_grad():
            l2 = (model(nodes_t) - u_ref_t).pow(2).mean().sqrt().item()
            hist['l2'].append(l2)
            # 分类准确率: sign(KAN(sample_xy)) vs y
            pred_sample = model(sample_xy_t).squeeze()
            pseudo_acc = ((pred_sample * y_t) > 0).float().mean().item()
            hist['acc'].append(pseudo_acc)
        model.train()
        print(f'epoch {epoch}: loss={loss.item():.4e} pde={loss_pde.item():.2e} preset={loss_preset.item():.2e} L2={l2:.4e} pseudo_acc={pseudo_acc:.4f}')

elapsed = time.time() - t0
print(f'\n训练完成:  CIM调用={cim_count}次')


=== KAN + PDE + CIM 实时训练 (每50 epoch 刷新量子约束) ===
epoch 0: loss=1.3229e+05 pde=1.32e+05 preset=4.15e+01 L2=1.1068e+01 pseudo_acc=0.0935
  epoch 50: 调用 CIM 刷新量子约束 (第 2/10 次)...
    完成, RMSE vs 经典=9.0880e+00,
  epoch 100: 调用 CIM 刷新量子约束 (第 3/10 次)...
    完成, RMSE vs 经典=4.8929e+00,
epoch 100: loss=1.5015e+04 pde=1.33e+04 preset=1.20e+02 L2=1.1896e+01 pseudo_acc=0.9375
  epoch 150: 调用 CIM 刷新量子约束 (第 4/10 次)...
    完成, RMSE vs 经典=9.7844e+00,
  epoch 200: 调用 CIM 刷新量子约束 (第 5/10 次)...
    完成, RMSE vs 经典=5.5309e+00,
epoch 200: loss=2.0624e+03 pde=4.91e+02 preset=5.59e+01 L2=1.0895e+01 pseudo_acc=0.9280
  epoch 250: 调用 CIM 刷新量子约束 (第 6/10 次)...
    完成, RMSE vs 经典=8.7757e+00,
  epoch 300: 调用 CIM 刷新量子约束 (第 7/10 次)...
    完成, RMSE vs 经典=4.4173e+00,
epoch 300: loss=8.5643e+02 pde=1.33e+02 preset=2.48e+01 L2=7.4362e+00 pseudo_acc=0.9145
  epoch 350: 调用 CIM 刷新量子约束 (第 8/10 次)...
    完成, RMSE vs 经典=1.0070e+01,
  epoch 400: 调用 CIM 刷新量子约束 (第 9/10 次)...
    完成, RMSE vs 经典=6.4913e+00,
epoch 400: loss=4.4335e+02

In [15]:
# ========== 评估 & 保存 ==========
model.eval()
with torch.no_grad():
    u_kan = model(nodes_t).cpu().numpy().flatten()
    pred_sample = model(sample_xy_t).squeeze().cpu().numpy()

# 回归指标
l2_kan = np.sqrt(np.mean((u_kan - u_ref)**2))
rmse_kan = np.sqrt(np.mean((u_kan[internal] - u_ref_i)**2))

# 分类指标: sign(KAN(p_k)) vs y_k
pseudo_acc = np.mean((pred_sample * y_s) > 0)
pulsar_acc = np.mean((pred_sample[y_s==1] * y_s[y_s==1]) > 0)
rfi_acc = np.mean((pred_sample[y_s==-1] * y_s[y_s==-1]) > 0)
quantum_init_acc = np.mean((u_quantum_init[internal][np.searchsorted(np.where(internal)[0], np.arange(len(y_s)))] * y_s) > 0) if False else np.nan

print(f'\n========== 最终结果 ==========')
print(f'KAN+量子实时 vs FEM:  L2={l2_kan:.4e}, 内部RMSE={rmse_kan:.4e}')
print(f'Pseudo分类准确率: {pseudo_acc:.4f} (脉冲星={pulsar_acc:.4f}, RFI={rfi_acc:.4f})')
print(f'首次量子预设 RMSE vs 经典: {rmse_init:.6e}')
print(f'CIM 调用: {cim_count} 次, 各次 RMSE: {[f"{r:.4e}" for r in cim_rmse_history]}')


========== 最终结果 ==========
KAN+量子实时 vs FEM:  L2=4.2824e+00, 内部RMSE=4.3972e+00
Pseudo分类准确率: 0.9075 (脉冲星=0.0107, RFI=1.0000)
首次量子预设 RMSE vs 经典: 1.033250e+01
CIM 调用: 10 次, 各次 RMSE: ['1.0332e+01', '9.0880e+00', '4.8929e+00', '9.7844e+00', '5.5309e+00', '8.7757e+00', '4.4173e+00', '1.0070e+01', '6.4913e+00', '8.8549e+00']


In [16]:
# ========== 保存所有结果 ==========
np.save(f'{OUTPUT_DIR}/htru2_u_kan_quantum.npy', u_kan)
np.save(f'{OUTPUT_DIR}/htru2_u_ref.npy', u_ref)
np.save(f'{OUTPUT_DIR}/htru2_u_quantum_init.npy', u_quantum_init)
np.save(f'{OUTPUT_DIR}/htru2_nodes.npy', nodes)
np.save(f'{OUTPUT_DIR}/htru2_internal.npy', internal)
np.save(f'{OUTPUT_DIR}/htru2_X_2d.npy', X_2d)
np.save(f'{OUTPUT_DIR}/htru2_y_s.npy', y_s)
np.save(f'{OUTPUT_DIR}/htru2_K_ii.npy', K_ii)
np.save(f'{OUTPUT_DIR}/htru2_rhs.npy', rhs)

results = {
    'method': 'kan_pde_energy_qubo_realtime',
    'qubo_type': 'energy_functional',
    'dataset': 'HTRU2_pulsar',
    'telescope': 'Parkes_64m_radio',
    'grid': f'{N}x{N}',
    'n_internal': int(n_i),
    'qubo_bits': int(nvar),
    'kappa_K': float(cond_K),
    'epochs': N_epoch,
    'cim_interval': CIM_INTERVAL,
    'cim_calls': cim_count,
    'cim_epochs': hist['cim_epochs'],
    'cim_rmse_history': cim_rmse_history,
    'total_time_sec': float(elapsed),
    'L2_vs_FEM': float(l2_kan),
    'RMSE_internal': float(rmse_kan),
    'pseudo_accuracy': float(pseudo_acc),
    'pulsar_accuracy': float(pulsar_acc),
    'rfi_accuracy': float(rfi_acc),
    'init_quantum_rmse': float(rmse_init),
    'loss_history': hist['loss'],
    'l2_history': hist['l2'],
    'acc_history': hist['acc'],
}
with open(f'{OUTPUT_DIR}/result_kan_quantum_realtime.json', 'w') as f:
    json.dump(results, f, indent=2)
print('全部结果已保存')

全部结果已保存
